# Interactive IQ sweep fitter with `pipeline_v2`

This notebook shows how to use `citkid.pipeline_v2.interactive.run_sweep_fitter` to fit IQ loops across a sweep parameter such as power, temperature, or field.

Each sweep point gets its own `DataSet` and `AnalysisRunner`, but they all share one interactive window. The window lets you move through resonators and sweep points, rerun panels, save results, and mark data as bad.

---

## What sweep fitter expects

`run_sweep_fitter(...)` needs:

1. A function `make_custom_steps(sweep_idx)` that returns the calibration loading steps for one sweep index.
2. A calibration YAML and analysis YAML that describe the IQ analysis pipeline.
3. An output zarr root where each sweep point is stored in its own subgroup.
4. A per-resonator x value and y value for the left-hand sweep scatter plot.

`pipeline_v2` safety rules still apply inside sweep fitter:

- Running one panel invalidates later panels for the current `(sweep_idx, data_idx)` selection.
- Later panels are cleared and marked as needing a rerun rather than being silently recomputed.
- The UI asks before you save or navigate away while downstream panels still need a rerun.
- Leading global/global-res analysis steps are only run automatically when their outputs do not already exist for that sweep runner.

In [ ]:
import numpy as np
import zarr
from citkid.pipeline_v2.framework import plStep
from citkid.pipeline_v2.interactive import run_sweep_fitter

# ---------------------------------------------------------------------
# Raw input and output stores
# ---------------------------------------------------------------------

# Raw data source (read-only)
root_raw = zarr.open(
    'path/to/raw_data.zarr',
    mode='r',
)

# Output root. sweep_fitter creates/uses subgroups like
# sweep_000, sweep_001, ... inside this store.
root_out = zarr.open(
    'path/to/output_sweep_fitter.zarr',
    mode='a',
)

# Number of sweep points and resonators
n_sweep = ...   # int
nrows   = ...   # int

## Define the per-sweep calibration loading steps

`make_custom_steps(sweep_idx)` must return the calibration steps for exactly one sweep point. These steps usually load:

- one shared global block (`nrows`, `fres_all`, `qres_all`, ...)
- one global-res block (`fres`, `qres`, `ares`, `res_idxs`, ...)
- per-row fine sweep data (`ff`, `zf`)
- per-row gain sweep data (`fg`, `zg`)

If a value should differ by resonator, make it `per-row` or `vectorized`, not `global`.

In [ ]:
def make_custom_steps(sweep_idx):
    """Return the calibration-loading steps for one sweep index."""

    def load_global_data():
        # Example: values shared across all resonators for this sweep index
        fres_all = ...
        qres_all = ...
        return fres_all, qres_all, nrows

    def load_global_res_data():
        # Example: one value per resonator, computed once for all rows
        fres = ...
        qres = ...
        ares = ...      # often the x-axis value for the sweep plot
        res_idxs = ...
        return fres, qres, ares, res_idxs

    def load_data_f(data_idx):
        # Fine sweep for one resonator. Frequencies should be ascending.
        ff = ...
        zf = ...
        return ff, zf

    def load_data_g(data_idx):
        # Gain sweep for one resonator. Frequencies should be ascending.
        fg = ...
        zg = ...
        return fg, zg

    custom_steps = [
        plStep('load_global_data', load_global_data, [], ['fres_all', 'qres_all', 'nrows'], 'global'),
        plStep('load_global_res_data', load_global_res_data, [], ['fres', 'qres', 'ares', 'res_idxs'], 'global-res'),
        plStep('load_data_f', load_data_f, ['data_idx'], ['ff', 'zf'], 'per-row'),
        plStep('load_data_g', load_data_g, ['data_idx'], ['fg', 'zg'], 'per-row'),
    ]
    return custom_steps

In [ ]:
def y_func(AR, data_idx):
    """Return one scalar y value for the sweep scatter plot."""
    try:
        iq_popt = np.asarray(AR.DS.iq_popt[data_idx], dtype=np.float64)
        if not np.all(np.isfinite(iq_popt)):
            return None
        # Example: the nonlinearity parameter a
        return float(iq_popt[4])
    except Exception:
        return None

## Choose the scatter-plot axes

The sweep window needs one x value and one y value for each `(sweep_idx, data_idx)` pair.

- `x_param_name` is a `DataSet` attribute loaded from the runner, usually something like `ares` or `fres`.
- `y_func(AR, data_idx)` is a callable that extracts one scalar from the currently fitted outputs for the left-hand scatter plot.

If the y value is not available yet, return `None`.

In [ ]:
run_sweep_fitter(
    make_custom_steps = make_custom_steps,
    cal_yaml_path = 'iq',
    analysis_yaml_path = 'iq',
    root = root_out,
    n_sweep = n_sweep,
    x_param_name = 'ares',   # DS attribute used on the x-axis
    x_name = 'Power (uW)',
    y_func = y_func,         # callable: y_func(AR, data_idx) -> scalar | None
    y_name = 'Nonlinearity a',
    start_sweep_idx = 0,
    start_data_idx = 0,
)

## Practical notes

- Sweep fitter saves panel outputs when you explicitly save, when you change selection, or when the window closes.
- Re-running an earlier panel marks later panels stale for the current selection. The window clears those plots and asks before you leave that stale state.
- Background prefetch is read-only. It prepares plot caches for the next resonator but does not execute pipeline steps or save anything.
- Leading global/global-res analysis steps are initialized once per sweep runner when needed. If they already exist in zarr, sweep fitter loads them instead of rerunning them.
- If you need to deliberately rerun a global/global-res analysis step after outputs already exist, do that from code with `allow_global_step_overwrite=True`; the UI is intentionally conservative by default.